# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# --- setup: clone repo in Colab, put work/src on the path, build (or load cached) cutoff frames ---
import os, sys, json, datetime as dt
REPO_DIR = "/content/FlyRank-ML-Internship"
if os.path.isdir("/content") and not os.environ.get("CAPSTONE_LOCAL"):
    if not os.path.isdir(REPO_DIR):
        os.system(f"git clone https://github.com/joannadulay/FlyRank-ML-Internship.git {REPO_DIR}")
    os.chdir(REPO_DIR)
    os.system("pip install -q datasets duckdb scikit-learn huggingface_hub pyarrow")
sys.path.insert(0, "work/src")
import numpy as np, pandas as pd
import capstone_lib as L
os.makedirs(L.OUT_DIR, exist_ok=True)
plan, frames, TOKEN, LOCAL = L.start()
train, val = frames["train"], frames["val"]          # the sealed test frame is not touched in this notebook
print("cutoffs  train:", [str(t) for t in plan["train"]], "| val:", plan["val"])
print("rows     train:", len(train), "| val:", len(val))

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 72.0MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 86.2MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

... (13 outputs left out)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

cutoffs  train: ['2026-01-11', '2026-02-08', '2026-03-08', '2026-04-05'] | val: 2026-05-03
rows     train: 237571 | val: 85997


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [25]:
# FINDING 1 (Finding #4, "The Freshness Multiplier", p. 9)
# Claim: pages older than 365 days that were refreshed within 30 days show a 3.2x health boost (10.7 -> 34.5) and 57x more impressions (71 -> 4,039); "refresh timing is one of the strongest measured levers".
# Label: the outcomes are the health score and average impressions. Health score is a composite built partly from impressions (30 pts), position (30), CTR (20) and scroll depth (20), so "3.2x health" and "57x impressions" are not two independent results. The groups come from a metadata field (days since last update), not from an experiment.
# Validation: the paper doesn't say whether this compares the same pages before and after a refresh, or two different groups of pages at one point in time. It reports no confidence intervals or p-values and doesn't give group sizes for this cell. The sample is limited to pages with impressions and sessions above zero, and the paper already warns about survivor bias in a nearby cell.
# Why it matters: if teams tend to refresh pages that already have demand (the paper's own playbook recommends refreshing pages with proven visibility), the refreshed group would show higher impressions even without a refresh effect. Average impressions are also dominated by a few very large pages (in my own data the largest page had about 800x the median).
# Question for the authors: could the comparison be repeated on the same pages before and after the update, or against a matched group of pages with similar earlier impressions, and could it report medians as well as means?

# FINDING 2 (ML appendix, "What Predicts Growth?", p. 29)
# Claim: a logistic regression separates growing from declining pages with 71% holdout accuracy.
# Label: "growing" and "declining" come from a rule applied to the same data: the change in impressions between the last 30 days and the previous 30 days (up or down by more than 10%, p. 5).
# Validation: the paper describes an 80/20 split but doesn't say whether it was by time or by brand, so pages from the same brand could sit on both sides. Features include recent impressions and days visible, and the text says these are among the strongest signals; both come from the same impression history that defines the label, so the features may overlap the label window. The paper gives accuracy only, with no baseline: in the portfolio-wide direction table (p. 6), about 62% of the up and down pages are growing (74.8K of 120.4K), so always guessing "growing" could score near 62% if the ML sample is similar. It also doesn't report the class mix of the 61.8K-page sample.
# Link to my own audit: my random split scored 0.534 against 0.438 under a time-aware split, so a split that lets the same pages sit on both sides inflated my result by about 0.09.
# Question for the authors: were all features computed only from days before the 30-day label window, was the 71% measured on a split by time or by brand, and how does it compare with always predicting the majority class?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [23]:
from sklearn.model_selection import train_test_split
choice = json.load(open("work/outputs/model_choice.json"))
mk = lambda: L.make_models()[choice["model"]]
F = L.FEATURES
audit = {}

# (a) random rows from the pooled development data: the same page shows up at several cutoffs on both sides
pool = pd.concat([train, val], ignore_index=True)
samp = pool.sample(min(400_000, len(pool)), random_state=L.CFG["seed"])
a_tr, a_te = train_test_split(samp, test_size=0.3, random_state=L.CFG["seed"], stratify=samp.y)
audit["random_split"] = L.score(a_te.y, mk().fit(a_tr[F], a_tr.y).predict(a_te[F]))

# (b) time-aware: exactly the Week-5 design
audit["time_split"] = L.score(val.y, mk().fit(train[F], train.y).predict(val[F]))

# (c) time-aware + unseen clients: hold out 20% of clients from training entirely
rng = np.random.default_rng(L.CFG["seed"])
clients = np.sort(val.client.unique())
held = set(rng.choice(clients, size=max(1, int(0.2 * len(clients))), replace=False))
tr_c, va_c = train[~train.client.isin(held)], val[val.client.isin(held)]
audit["time_and_client_split"] = L.score(va_c.y, mk().fit(tr_c[F], tr_c.y).predict(va_c[F]))

print(pd.Series({k: v["macro_f1"] for k, v in audit.items()}).round(3).to_string())
print(f"\nrandom-split minus time-aware macro F1: {audit['random_split']['macro_f1'] - audit['time_split']['macro_f1']:+.3f}")
print("held-out clients:", len(held), "| rows:", len(va_c))

random_split             0.531
time_split               0.439
time_and_client_split    0.431

random-split minus time-aware macro F1: +0.092
held-out clients: 7 | rows: 14030


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [24]:
# (1) future-deletion test at the validation cutoff
fbm = L.find_month_files(TOKEN, LOCAL)
T = plan["val"]
months = sorted({d.strftime("%Y-%m") for d in pd.date_range(T - dt.timedelta(days=55), T)})
vf = L.verify_no_future(L.fetch_paths(fbm, months, TOKEN, LOCAL), T)
print("Pages compared:", vf["n_both"], "| max abs difference across all features:", vf["max_abs_diff"])
assert vf["max_abs_diff"] < 1e-9, "a feature changed when future rows were removed -> leakage"

# (2) single-feature scan on the validation set
from sklearn.tree import DecisionTreeClassifier
tr_s = train.sample(min(200_000, len(train)), random_state=L.CFG["seed"])
single = {}
for f in F:
    t = DecisionTreeClassifier(max_depth=3, random_state=L.CFG["seed"]).fit(tr_s[[f]].fillna(-1e9), tr_s.y)
    single[f] = L.score(val.y, t.predict(val[[f]].fillna(-1e9)))["macro_f1"]
single = pd.Series(single).sort_values(ascending=False)
print("\nSingle-feature macro F1 (depth-3 tree, validation):\n", single.round(3).to_string())
print("flag threshold 0.90 ->", "RED FLAG" if single.max() > 0.9 else "none above 0.90")

# (3) identifiers are not features
assert not {"client", "content", "cutoff"} & set(F), "identifier or date in the feature list"
# (4) how much of validation is pages the model already saw in training
key = lambda d: d.client + "|" + d.content
seen = key(val).isin(set(key(train))).mean()
print(f"\nShare of validation pages that also appear in training (earlier cutoffs): {seen:.1%}  (expected for a forward-in-time split)")

audit["leakage"] = dict(max_abs_diff=vf["max_abs_diff"], single_feature_max=float(single.max()),
                        single_feature_top=single.index[0], val_pages_seen_in_train=float(seen))
json.dump(audit, open("work/outputs/w06_audit.json", "w"), indent=1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages compared: 88211 | max abs difference across all features: 1.3322676295501878e-15

Single-feature macro F1 (depth-3 tree, validation):
 r7                0.400
cv_B              0.269
d_impr            0.247
avail_share_B     0.192
d_ctr             0.152
org_per_day_B     0.139
d_pos             0.139
pos_B             0.138
ctr_B             0.134
log_rate_B        0.134
active_share_B    0.134
ga4_cov_B         0.134
eng_per_day_B     0.134
flag threshold 0.90 -> none above 0.90

Share of validation pages that also appear in training (earlier cutoffs): 80.0%  (expected for a forward-in-time split)


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [21]:
import os, json

# Reload the saved audit if the runtime restarted and `audit` is gone
if "audit" not in globals():
    audit = json.load(open("work/outputs/w06_audit.json"))

ts, rs = audit["time_split"]["macro_f1"], audit["random_split"]["macro_f1"]
choice = json.load(open("work/outputs/model_choice.json"))
try:
    dev = json.load(open("work/outputs/w05_dev_metrics.json"))
    bs = dev["val_scores"]["baseline:" + choice["baseline"]]["macro_f1"]
    vs = f", compared with {bs:.3f} for the strongest simple rule ({choice['baseline'].replace('_', ' ')})"
except (FileNotFoundError, KeyError):
    vs = ""
print("BOLD : The model predicts which pages will decline.")
print(f"SAFE : On held-out pages from a later month, a model using the previous 56 days of search signals reached a measured macro F1 of {ts:.3f} "
      f"under a time-aware split ({rs:.3f} under a random split, which overstates it){vs}. "
      "This is an observed, directional association that can help rank pages for human review; it is decision-support, not a causal or guaranteed prediction.")

BOLD : The model predicts which pages will decline.
SAFE : On held-out pages from a later month, a model using the previous 56 days of search signals reached a measured macro F1 of 0.439 under a time-aware split (0.531 under a random split, which overstates it), compared with 0.450 for the strongest simple rule (momentum tree). This is an observed, directional association that can help rank pages for human review; it is decision-support, not a causal or guaranteed prediction.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.